# 05. <span style="color:#7CDBA0">Dataset</span> и <span style="color:#6CB6FF">DataLoader</span> — ч.2 (практика)

Видео: [5. Подготовка данных ч.2. <span style="color:#7CDBA0">Dataset</span> и <span style="color:#6CB6FF">DataLoader</span>](https://www.youtube.com/watch?v=ztTlRgwdhkU&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=6)

## Зачем этот ноутбук (не в вакууме)

Обучение сети — цикл примерно такой:

```text
сырые данные на диске
    → Dataset          (как достать один пример: картинка + метка)
    → DataLoader       (нарезать на батчи, перемешать)
    → for xb, yb in loader:
          модель(xb) → loss → backward → обновить веса
```

Видео 5 у автора — **первый кирпич** его проекта на MNIST: свой <span style="color:#7CDBA0">Dataset</span> + <span style="color:#6CB6FF">DataLoader</span>. Без этого дальше (модель, цикл обучения) не к чему подключаться.

Здесь то же самое, но на маленьком наборе «фрукты» (создаёт ячейка ниже). Потом перенесёшь каркас в `project/fashion_mnist.ipynb` — уже на настоящих данных.

**Как решать.** Замени `...` на код → запусти проверку. Решения — в `05_dataset_dataloader_2_решения.ipynb`, только после своей попытки.

| блок | о чём | где в видео |
|:---|:---|:---|
| 1 | разминка: <span style="color:#7CDBA0">Dataset</span> на тензорах + <span style="color:#6CB6FF">DataLoader</span> | 2:36–3:08, 14:38 |
| 2 | свой <span style="color:#7CDBA0">FruitsDataset</span> по папкам (как у автора на MNIST) | 2:36–9:56 |
| 3 | train / val и три <span style="color:#6CB6FF">DataLoader</span> | 14:07–15:41 |
| 4 | тот же каркас на кликах (рекомендации) | — |

In [16]:
import os
import json
import shutil

import numpy as np
import torch
from PIL import Image
from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split,
    RandomSampler,
    SequentialSampler,
)

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

## 0. Данные

Ячейка ниже создаёт рядом с ноутбуком папку `_data/`:

```text
_data/
├── fruits/
│   ├── train/
│   │   ├── README.txt          ← мусор, не класс
│   │   ├── apple/  12 картинок
│   │   ├── pear/   10 картинок + notes.txt (мусор)
│   │   └── plum/    3 картинки  ← мало!
│   └── test/
│       ├── apple/ pear/ plum/  по 4 картинки
└── clicks.json                 ← для блока 4
```

Картинки 8×8, серые. У каждого класса своя яркость (apple тёмный, pear средний, plum светлый) — так проверки узнают, что метка совпадает с картинкой. Запусти ячейку и загляни в папку глазами.

In [17]:
DATA = "_data"
FRUITS = os.path.join(DATA, "fruits")
TRAIN_DIR = os.path.join(FRUITS, "train")
TEST_DIR = os.path.join(FRUITS, "test")
CLICKS_PATH = os.path.join(DATA, "clicks.json")

SIZE = 8
BRIGHTNESS = {"apple": 40, "pear": 120, "plum": 200}
COUNTS = {
    "train": {"apple": 12, "pear": 10, "plum": 3},
    "test": {"apple": 4, "pear": 4, "plum": 4},
}


def make_data():
    shutil.rmtree(DATA, ignore_errors=True)
    rng = np.random.default_rng(0)
    for split, per_class in COUNTS.items():
        for name, n in per_class.items():
            folder = os.path.join(FRUITS, split, name)
            os.makedirs(folder)
            for k in range(n):
                noise = rng.integers(-20, 21, size=(SIZE, SIZE))
                img = np.clip(BRIGHTNESS[name] + noise, 0, 255).astype(np.uint8)
                Image.fromarray(img).save(os.path.join(folder, f"{name}_{k:03d}.png"))
    with open(os.path.join(TRAIN_DIR, "README.txt"), "w", encoding="utf-8") as f:
        f.write("описание датасета")
    with open(os.path.join(TRAIN_DIR, "pear", "notes.txt"), "w", encoding="utf-8") as f:
        f.write("не картинка")

    clicks = [
        {"user": "u_17", "item": "sku_501", "click": 1},
        {"user": "u_17", "item": "sku_777", "click": 0},
        {"user": "u_03", "item": "sku_501", "click": 1},
        {"user": "u_42", "item": "sku_123", "click": 1},
        {"user": "u_42", "item": "sku_501", "click": 0},
        {"user": "u_03", "item": "sku_777", "click": 1},
        {"user": "u_99", "item": "sku_123", "click": 0},
        {"user": "u_17", "item": "sku_123", "click": 1},
    ]
    with open(CLICKS_PATH, "w", encoding="utf-8") as f:
        json.dump(clicks, f, ensure_ascii=False, indent=2)


make_data()
print(sorted(os.listdir(TRAIN_DIR)))
print(sorted(os.listdir(os.path.join(TRAIN_DIR, "pear")))[:3], "...")

['README.txt', 'apple', 'pear', 'plum']
['notes.txt', 'pear_000.png', 'pear_001.png'] ...


## 1. Разминка: свой <span style="color:#7CDBA0">Dataset</span> на тензорах
*видео 2:36–3:08*

**Зачем.** <span style="color:#6CB6FF">DataLoader</span> не умеет сам брать «признаки из одного тензора, метки из другого». Ему нужен **один объект**, у которого:
- `len(ds)` — сколько примеров;
- `ds[i]` — один пример сразу парой `(x, y)`.

Такой объект — наследник `torch.utils.data.Dataset` с тремя методами:

| метод | что делает |
|:---|:---|
| <span style="color:#C9A0DC">__init__</span>(...) | запоминает, **откуда** брать примеры (тензоры, пути к файлам, transform) |
| <span style="color:#C9A0DC">__len__</span>(self) | сколько примеров — по нему <span style="color:#6CB6FF">DataLoader</span> считает число батчей |
| <span style="color:#C9A0DC">__getitem__</span>(self, idx) | возвращает **один** пример: пару `(данные, ответ)` |

`ds[i]` вызывает `ds.__getitem__(i)`, `len(ds)` — `ds.__len__()`. Сами по себе у обычного класса Python этих возможностей нет — их дописывают вручную (как у `list` они уже есть внутри).

### Зачем сначала <span style="color:#7CDBA0">ToyDataset</span>, потом фрукты (конспект)

| | блок 1 <span style="color:#7CDBA0">ToyDataset</span> | блок 2 <span style="color:#7CDBA0">FruitsDataset</span> / клики |
|:---|:---|:---|
| откуда данные | сэмулировали: тензоры `X_all`, `y_all` уже в памяти | тянем с диска: папки png / `clicks.json` |
| что усложнили | почти ничего: `return self.X[idx], self.y[idx]` | обход папок, фильтр `.txt`, открыть файл → тензор |
| **суть** | <span style="color:#7CDBA0">Dataset</span> = **один** пример `(x, y)` | **та же** |
| дальше | <span style="color:#6CB6FF">DataLoader</span> = **батч** | **то же** |

Мы не учили два разных мира. Сначала на игрушечных признаках и метках отработали каркас (чтобы не путаться в файлах). Потом подставили реальные пути — задание сложнее, роли классов те же.

Почему так: <span style="color:#6CB6FF">DataLoader</span> всегда работает одинаково — вызывает `ds[i]`. Ему всё равно, тензор это в RAM или png с диска. Меняется только внутренность <span style="color:#C9A0DC">__getitem__</span>.

> Готовый аналог для тензоров в памяти: <span style="color:#7CDBA0">TensorDataset</span>. Свой класс пишут, когда данные нестандартные (папки, json, CSV) — как у автора и как будет у тебя дальше.

**1.1** Допиши <span style="color:#7CDBA0">ToyDataset</span>: хранит `X` формы `(N, F)` и `y` формы `(N,)`; `ds[i]` → `(X[i], y[i])`.

**Ожидаемый вывод:**

```text
len(toy_ds): 20
x0.shape:    torch.Size([8])   ← один пример — вектор из 8 признаков
y0:          tensor(...)       ← метка класса, число от 0 до 4
```

In [35]:
X_all = torch.randn(20, 8)  # 20 примеров по 8 признаков
# print("🟢🔵🔴 ~ X_all:", X_all)
y_all = torch.randint(0, 5, (20,))  # 5 классов
# print("🟢🔵🔴 ~ y_all:", y_all)

In [38]:
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __repr__(self):
        return (
            f"ToyDataset(n={len(self)}, "
            f"features={self.X.shape[1]}, "
            f"X={tuple(self.X.shape)}, y={tuple(self.y.shape)})"
        )


toy_ds = ToyDataset(X_all, y_all)
x0, y0 = toy_ds[0]
print(toy_ds)
print(len(toy_ds), x0.shape, y0)

ToyDataset(n=20, features=8, X=(20, 8), y=(20,))
20 torch.Size([8]) tensor(4)


In [20]:
need(isinstance(toy_ds, Dataset), "класс должен наследоваться от Dataset")
need(len(toy_ds) == 20, f"len={len(toy_ds)}, ждали 20")
need_tensor(x0, "x0")
need(x0.shape == (8,), f"x0.shape={tuple(x0.shape)}")
need(
    torch.equal(x0, X_all[0]) and int(y0) == int(y_all[0]),
    "toy_ds[0] должен совпадать с X_all[0], y_all[0]",
)
x19, y19 = toy_ds[19]
need(
    torch.equal(x19, X_all[19]) and int(y19) == int(y_all[19]),
    "проверь крайний индекс 19",
)
print("1.1 — верно")

1.1 — верно


**Теория. <span style="color:#6CB6FF">DataLoader</span> — зачем поверх <span style="color:#7CDBA0">Dataset</span>.**

### Связка (конспект)

<span style="color:#6CB6FF">DataLoader</span> **сам** много раз вызывает <span style="color:#7CDBA0">Dataset</span>, чтобы набрать батч. В загрузчик передают **объект** датасета, а не уже загруженные картинки:

```text
Dataset          — умеет: len(ds) и ds[i] → один (x, y)
       ↑ вызывает ds[i] несколько раз
DataLoader       — собирает батч (xb, yb), умеет shuffle
       ↓
for xb, yb in loader:   — сюда уже приходят пачки
```

Пример при `batch_size=6`:

```text
ds[i1], ds[i2], ds[i3], ds[i4], ds[i5], ds[i6]
              ↓ stack
         один батч (xb, yb)
```

Поэтому свой <span style="color:#7CDBA0">Dataset</span> и пишут: у <span style="color:#6CB6FF">DataLoader</span> нет другого способа узнать, как открыть png/json. Ему нужен единый контракт «дай пример номер i».

<span style="color:#7CDBA0">Dataset</span> умеет только «дай пример номер i». Сеть учат не по одному примеру, а **батчами** (16 / 32 / 64 сразу). <span style="color:#6CB6FF">DataLoader</span>:
1. берёт `batch_size` индексов;
2. для каждого вызывает `ds[i]` (твой <span style="color:#C9A0DC">__getitem__</span>);
3. склеивает результаты по новой оси 0 — как `torch.stack` из ноутбука 03/04;
4. при `shuffle=True` на каждой эпохе меняет порядок (чтобы не выучить последовательность).

```python
loader = DataLoader(ds, batch_size=4, shuffle=True)
for xb, yb in loader:   # xb: (4, F), yb: (4,)  — весь проход = одна эпоха
    ...
```

Автор в конце видео 5 делает то же для MNIST: три загрузчика train / val / test, потом в цикле обучения `for xb, yb in train_loader`.

`next(iter(loader))` — взять **один** батч без цикла: удобно проверить форму, не гоняя весь датасет.

**1.2** Собери `toy_loader` (`batch_size=4`, `shuffle=True`) и возьми из него один батч `xb`, `yb`.

**Ожидаемый вывод:**

```text
xb.shape: torch.Size([4, 8])   ← батч: 4 примера × 8 признаков
yb.shape: torch.Size([4])      ← 4 метки
```

In [39]:
toy_loader = DataLoader(toy_ds, batch_size=4, shuffle=True)
for xb, yb in toy_loader:
    xb, yb = next(iter(toy_loader))
print(xb.shape, yb.shape)

torch.Size([4, 8]) torch.Size([4])


In [22]:
need(isinstance(toy_loader, DataLoader), "нужен DataLoader")
need(toy_loader.batch_size == 4, f"batch_size={toy_loader.batch_size}")
need(isinstance(toy_loader.sampler, RandomSampler), "для train нужен shuffle=True")
need_tensor(xb, "xb")
need(xb.shape == (4, 8), f"xb.shape={tuple(xb.shape)}, ждали (4, 8)")
need(yb.shape == (4,), f"yb.shape={tuple(yb.shape)}, ждали (4,)")
print("1.2 — верно")

1.2 — верно


## 2. Свой <span style="color:#7CDBA0">FruitsDataset</span>
*видео 5:45–9:56*

**Зачем.** У автора картинки лежат в папках-классах на диске (MNIST). Их нельзя один раз загрузить все в RAM и забыть: датасет большой. Поэтому:
- в <span style="color:#C9A0DC">__init__</span> собираем **только пути** и метки;
- в <span style="color:#C9A0DC">__getitem__</span> читаем **один** файл, когда его попросил <span style="color:#6CB6FF">DataLoader</span>.

Тот же каркас, что в разминке — меняется только «откуда брать x». Это ровно то, что автор пишет в видео 5, только у тебя «фрукты» вместо цифр (и лишние `.txt`, которые надо пропустить).

Нужные функции:

```python
os.listdir(folder)                 # имена всего внутри папки
os.path.isdir(path)                # это папка?
os.path.join(folder, name)         # склеить путь
with Image.open(path) as im:       # PIL; with — чтобы файл сразу закрылся
    arr = np.array(im)             # (H, W), uint8, значения 0..255
torch.tensor(arr, dtype=torch.float32)
```

**2.1** Напиши `FruitsDataset(root, transform=None)`. Требования (часть — **не как у автора**):

| что | требование |
|:---|:---|
| `self.classes` | **отсортированный** список имён папок-классов (`README.txt` — не класс) |
| `self.class_to_idx` | имя → индекс по порядку `classes` |
| `self.data_list` | пары `(путь, индекс_класса)`, только `.png` |
| <span style="color:#C9A0DC">__getitem__</span> | возвращает `(x, label)`: `x` — **тензор** `float32` формы `(1, H, W)` со значениями 0..1, `label` — `int` |
| `transform` | если задан — применить к `x` перед возвратом |

Подсказка к форме: `(H, W)` → `(1, H, W)` — это `unsqueeze(0)` (ось каналов, как в видео 4).

**Ожидаемый вывод** (подписи — что означает каждое значение):

```text
classes:       ['apple', 'pear', 'plum']
class_to_idx:  {'apple': 0, 'pear': 1, 'plum': 2}
len(train_ds): 25          ← 12 apple + 10 pear + 3 plum (без .txt)
len(test_ds):  12
x.shape:       torch.Size([1, 8, 8])   ← канал × высота × ширина
x.dtype:       torch.float32
label:         0           ← индекс класса apple (первый в classes)
```

In [ ]:
class FruitsDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        # только папки-классы (README.txt отсеется)
        self.classes = sorted(
            name for name in os.listdir(root) if os.path.isdir(os.path.join(root, name))
        )
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}
        self.transform = transform

        self.data_list = []
        for name in self.classes:
            folder = os.path.join(root, name)
            for file_name in sorted(os.listdir(folder)):
                if file_name.endswith(".png"):  # notes.txt пропускаем
                    path = os.path.join(folder, file_name)
                    self.data_list.append((path, self.class_to_idx[name]))

    def __len__(self):
        return len(self.data_list)  # 25 для train

    def __getitem__(self, idx):
        path, label = self.data_list[idx]
        with Image.open(path) as im:
            arr = np.array(im)  # (H, W), uint8, 0..255
            x = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
            x = x / 255
            if self.transform is not None:
                x = self.transform(x)
            return x, label


train_ds = FruitsDataset(TRAIN_DIR)
train_ds = FruitsDataset(TRAIN_DIR)

test_ds = FruitsDataset(TEST_DIR)
x, label = train_ds[0]
print("classes:      ", train_ds.classes)
print("class_to_idx: ", train_ds.class_to_idx)
print("len(train_ds):", len(train_ds))
print("len(test_ds): ", len(test_ds))
print("x.shape:      ", x.shape)
print("x.dtype:      ", x.dtype)
print("label:        ", label)

🟢🔵🔴 ~ train_ds: <__main__.FruitsDataset object at 0x0000019689382F90>
classes:       ['apple', 'pear', 'plum']
class_to_idx:  {'apple': 0, 'pear': 1, 'plum': 2}
len(train_ds): 25
len(test_ds):  12
x.shape:       torch.Size([1, 8, 8])
x.dtype:       torch.float32
label:         0


In [24]:
need(isinstance(train_ds, Dataset), "наследуйся от Dataset")
need(
    train_ds.classes == ["apple", "pear", "plum"],
    f"classes={train_ds.classes}: отсортируй и выкинь не-папки",
)
need(
    train_ds.class_to_idx == {"apple": 0, "pear": 1, "plum": 2},
    f"class_to_idx={train_ds.class_to_idx}",
)
need(
    len(train_ds) == 25,
    f"len(train_ds)={len(train_ds)}, ждали 12+10+3=25 (notes.txt не картинка)",
)
need(len(test_ds) == 12, f"len(test_ds)={len(test_ds)}, ждали 12")
need(
    test_ds.class_to_idx == train_ds.class_to_idx,
    "у train и test должно быть одинаковое соответствие класс → индекс",
)
need(
    all(isinstance(p, str) and p.endswith(".png") for p, _ in train_ds.data_list),
    "в data_list должны быть пути к .png, а не сами картинки",
)

need_tensor(x, "x")
need(x.shape == (1, 8, 8), f"x.shape={tuple(x.shape)}, ждали (1, 8, 8)")
need(x.dtype == torch.float32, f"x.dtype={x.dtype}")
need(
    0.0 <= x.min().item() and x.max().item() <= 1.0,
    "значения должны быть в 0..1 (раздели на 255)",
)
need(isinstance(label, int), f"label должен быть int, а не {type(label).__name__}")

_levels = torch.tensor([40, 120, 200]) / 255
for _i in range(len(train_ds)):
    _x, _y = train_ds[_i]
    _guess = (_levels - _x.mean()).abs().argmin().item()
    need(
        _guess == _y,
        f"train_ds[{_i}]: метка {_y}, а по яркости картинка похожа на класс {_guess}",
    )
print("2.1 — верно")

2.1 — верно


## 3. <span style="color:#E6C07B">random_split</span> и три <span style="color:#6CB6FF">DataLoader</span>
*видео 14:07–15:41*

**Зачем.** Модель нельзя оценивать на тех же данных, на которых училась — получится завышенный результат. Нужны три набора:
- **train** — учимся, `shuffle=True`;
- **val** — смотрим качество по ходу обучения, веса не трогаем, `shuffle=False`;
- **test** — один раз в самом конце (у автора test уже отдельная папка).

Отдельного val у нас нет — отрезаем его от train (как автор):

```python
part_a, part_b = random_split(ds, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
```

- доли (или точные длины `[20, 5]`) должны давать в сумме весь датасет;
- `generator` с seed — чтобы разбиение было одинаковым при каждом запуске;
- результат — `Subset`: не копия данных, а «датасет + список индексов» (`part_a.indices`).

После этого три <span style="color:#6CB6FF">DataLoader</span> — и данные готовы к циклу обучения (видео 11). Сейчас цикл ещё не пишем: только подготавливаем «трубы».

**3.1** Раздели `train_ds` на `train_part` / `val_part` в пропорции 80 / 20 с seed 42.

**Ожидаемый вывод:**

```text
len(train_part): 20   ← 80% от 25
len(val_part):    5   ← 20% от 25
```

In [25]:
train_part, val_part = random_split(
    train_ds, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

print(len(train_part), len(val_part))

20 5


In [26]:
need(
    len(train_part) == 20 and len(val_part) == 5,
    f"длины {len(train_part)}, {len(val_part)}; ждали 20 и 5",
)

_a, _b = set(train_part.indices), set(val_part.indices)

need(not (_a & _b), "train и val пересекаются")

need(_a | _b == set(range(25)), "вместе train и val должны покрыть все 25 примеров")

_again, _ = random_split(
    train_ds, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

need(
    list(_again.indices) == list(train_part.indices),
    "передай generator с seed 42 — иначе разбиение каждый раз новое",
)

print("3.1 — верно")

3.1 — верно


**3.2** Сначала **в уме**, потом кодом.

При `batch_size=6` сколько будет батчей и какого размера последний для `train_part` (20), `val_part` (5), `test_ds` (12)? Заполни `pred` парами `(число_батчей, размер_последнего)`.

Зачем считать: в логах обучения часто смотрят «батч 187 / 750» — это просто `len(loader)`. Если `N` не делится на `B`, последний батч короче — это нормально, не баг.

Затем создай `train_loader`, `val_loader`, `test_loader` с `batch_size=6`: train перемешивается, val и test — нет.

**Ожидаемый вывод:**

```text
xb.shape: torch.Size([6, 1, 8, 8])  ← батч 6 картинок: B × C × H × W
yb.shape: torch.Size([6])          ← 6 меток
yb.dtype: torch.int64              ← так DataLoader склеивает int-метки
```

In [27]:
pred = {
    "train": (4, 2),  # 20 = 6 + 6 + 6 + 2
    "val": (1, 5),  # 5 < 6 → один батч из 5
    "test": (2, 6),  # 12 = 6 + 6
}

train_loader = DataLoader(train_part, batch_size=6, shuffle=True)
val_loader = DataLoader(val_part, batch_size=6, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=6, shuffle=False)


xb, yb = next(iter(train_loader))

print(xb.shape, yb.shape, yb.dtype)

torch.Size([6, 1, 8, 8]) torch.Size([6]) torch.int64


In [28]:
_loaders = {"train": train_loader, "val": val_loader, "test": test_loader}
for _name, _loader in _loaders.items():
    _sizes = [bx.shape[0] for bx, _ in _loader]
    need(
        pred[_name] == (len(_sizes), _sizes[-1]),
        f"{_name}: ты предсказал {pred[_name]}, на деле {(len(_sizes), _sizes[-1])}",
    )
need(
    isinstance(train_loader.sampler, RandomSampler), "train_loader: нужен shuffle=True"
)
need(
    isinstance(val_loader.sampler, SequentialSampler),
    "val_loader: перемешивать не нужно",
)
need(
    isinstance(test_loader.sampler, SequentialSampler),
    "test_loader: перемешивать не нужно",
)
need(
    xb.shape == (6, 1, 8, 8),
    f"xb.shape={tuple(xb.shape)}, ждали (6, 1, 8, 8) = (B, C, H, W)",
)
need(
    yb.shape == (6,) and yb.dtype == torch.int64,
    f"yb: {tuple(yb.shape)}, {yb.dtype}; int-метки DataLoader склеивает в int64",
)
print("3.2 — верно")

3.2 — верно


## 4. То же на кликах (рекомендации)
*не из видео — перенос на другой тип данных*

**Зачем.** Каркас <span style="color:#7CDBA0">Dataset</span> / <span style="color:#6CB6FF">DataLoader</span> не про картинки — про **любые** пары «вход → ответ». В рекомендациях пример = взаимодействие: пользователь + товар → клик. Те же три метода, другие поля внутри.

В `_data/clicks.json` — список событий `{"user": "u_17", "item": "sku_501", "click": 1}`. Отличия от фруктов:

- файл маленький — его можно прочитать целиком в <span style="color:#C9A0DC">__init__</span> (`json.load`). Ленивое чтение нужно, когда каждый пример — тяжёлый файл;
- id — строки (`"u_17"`), а `nn.Embedding` (будет позже) понимает только индексы `0..n-1`. Поэтому, как `class_to_idx`, строим `user2idx` и `item2idx` по **отсортированным** уникальным id.

Позже (после видео 11) на таком датасете можно будет учить эмбеддинги: большой скор = подходит.

**4.1** Напиши `ClicksDataset(path)`:

| что | требование |
|:---|:---|
| `self.user2idx`, `self.item2idx` | отсортированные уникальные id → `0..n-1` |
| <span style="color:#C9A0DC">__getitem__</span> | `((user_idx, item_idx), click)`: два тензора `int64` формы `()` и тензор `float32` формы `()` |

**Ожидаемый вывод:**

```text
{'u_03': 0, 'u_17': 1, 'u_42': 2, 'u_99': 3} {'sku_123': 0, 'sku_501': 1, 'sku_777': 2}
tensor([1, 1, 0, 2]) tensor([1, 2, 1, 0]) tensor([1., 0., 1., 1.])
```

In [43]:
class ClicksDataset(Dataset):
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            self.records = json.load(f)
        users = sorted({r["user"] for r in self.records})
        items = sorted({r["item"] for r in self.records})
        self.user2idx = {u: i for i, u in enumerate(users)}
        self.item2idx = {it: i for i, it in enumerate(items)}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        user = torch.tensor(self.user2idx[r["user"]], dtype=torch.int64)
        item = torch.tensor(self.item2idx[r["item"]], dtype=torch.int64)
        click = torch.tensor(r["click"], dtype=torch.float32)
        return (user, item), click


clicks_ds = ClicksDataset(CLICKS_PATH)
clicks_loader = DataLoader(clicks_ds, batch_size=4, shuffle=False)
(bu, bi), by = next(iter(clicks_loader))
print(clicks_ds.user2idx, clicks_ds.item2idx)
print(bu, bi, by)

{'u_03': 0, 'u_17': 1, 'u_42': 2, 'u_99': 3} {'sku_123': 0, 'sku_501': 1, 'sku_777': 2}
tensor([1, 1, 0, 2]) tensor([1, 2, 1, 0]) tensor([1., 0., 1., 1.])


In [30]:
need(len(clicks_ds) == 8, f"len={len(clicks_ds)}, ждали 8")
need(
    clicks_ds.user2idx == {"u_03": 0, "u_17": 1, "u_42": 2, "u_99": 3},
    f"user2idx={clicks_ds.user2idx}",
)
need(
    clicks_ds.item2idx == {"sku_123": 0, "sku_501": 1, "sku_777": 2},
    f"item2idx={clicks_ds.item2idx}",
)
(_u, _i), _c = clicks_ds[0]
need_tensor(_u, "user")
need(
    _u.shape == () and _u.dtype == torch.int64,
    f"user: форма {tuple(_u.shape)}, {_u.dtype}; ждали (), int64",
)
need(
    _i.dtype == torch.int64 and _c.dtype == torch.float32,
    f"item {_i.dtype}, click {_c.dtype}",
)
need(
    (_u.item(), _i.item(), _c.item()) == (1, 1, 1.0),
    "clicks_ds[0]: u_17 → 1, sku_501 → 1, click 1.0",
)
need(bu.tolist() == [1, 1, 0, 2], f"bu={bu.tolist()}")
need(bi.tolist() == [1, 2, 1, 0], f"bi={bi.tolist()}")
need(by.tolist() == [1.0, 0.0, 1.0, 1.0], f"by={by.tolist()}")
print("4.1 — верно")

4.1 — верно


## 5. Что унести из видео 5

### Главное про два класса

| класс | роль |
|:---|:---|
| <span style="color:#7CDBA0">Dataset</span> | по индексу отдать **один** пример `(x, y)` (<span style="color:#C9A0DC">__getitem__</span>) и сказать длину (<span style="color:#C9A0DC">__len__</span>) |
| <span style="color:#6CB6FF">DataLoader</span> | много раз вызвать `ds[i]`, склеить в **батч**, при желании перемешать |

В `DataLoader(...)` передают сам объект датасета (`train_ds` / `train_part`), а не список картинок. Загрузчик сам дергает датасет, когда идёт `for xb, yb in loader`.

Данные с диска **не** грузят все сразу: в <span style="color:#C9A0DC">__init__</span> — пути/метки, в <span style="color:#C9A0DC">__getitem__</span> — один файл по запросу. В RAM в основном текущий батч.

### Как устроен этот ноутбук

1. **<span style="color:#7CDBA0">ToyDataset</span>** — сэмулировали признаки и метки тензорами в памяти. Нужен, чтобы понять каркас без возни с файлами.
2. **<span style="color:#7CDBA0">FruitsDataset</span> / клики** — те же три метода, но данные уже из папок/json (как у автора на MNIST, только другие файлы).
3. Суть на всём ноутбуке одна: **<span style="color:#7CDBA0">Dataset</span> = один элемент**, **<span style="color:#6CB6FF">DataLoader</span> = батч**. Усложнение — только в том, *откуда* <span style="color:#C9A0DC">__getitem__</span> достаёт `x`.

### Чеклист

1. Свой <span style="color:#7CDBA0">Dataset</span>: в <span style="color:#C9A0DC">__init__</span> — пути и метки, в <span style="color:#C9A0DC">__getitem__</span> — чтение одного файла.
2. `classes` сортируем — тогда train и test получают одно и то же `class_to_idx`.
3. Мусорные файлы фильтруем (`endswith(".png")`, `os.path.isdir`), пути — через `os.path.join`.
4. <span style="color:#E6C07B">random_split</span> с `generator` — воспроизводимое train / val.
5. <span style="color:#6CB6FF">DataLoader</span>: train `shuffle=True`, val / test — `False`; последний батч может быть короче.
6. Тот же каркас работает для любых данных: картинки, клики, таблицы.

**Проверка, что можно идти дальше:** своими словами объяснить таблицу выше и уметь глянуть `ds[0]` и `next(iter(loader))` (формы).

### Дальше (видео 6) — тизер

Для раскладки «папка = класс» в PyTorch есть готовый <span style="color:#E6C07B">ImageFolder</span>: почти то же, что наш <span style="color:#7CDBA0">FruitsDataset</span>, но одной строкой. Мы писали класс вручную, чтобы понять, что он делает под капотом.

```python
from torchvision.datasets import ImageFolder
train_ds = ImageFolder(TRAIN_DIR)
```

Ограничение: только такая структура папок. Метки в json/CSV, клики, регрессия координат — снова свой <span style="color:#7CDBA0">Dataset</span> (это тоже будет в видео 6).

**Сразу после этого ноутбука** — свой проект: `project/fashion_mnist.ipynb`, раздел «Видео 5».